In [19]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [20]:
class LineRatePredictor(nn.Module):
    def __init__(self, num_features):
        super(LineRatePredictor, self).__init__()

        self.conv1 = nn.Conv1d(num_features, 32, kernel_size=3)
        self.conv2 = nn.Conv1d(32, 64, kernel_size=3)

        self.lstm = nn.LSTM(
            input_size=64,
            hidden_size=32,
            batch_first=True
        )

        self.fc1 = nn.Linear(32, 16)
        self.fc2 = nn.Linear(16, 1)

    def forward(self, x):
        # x shape: (batch, time, features)

        x = x.permute(0, 2, 1)       # → (batch, features, time)

        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))

        x = x.permute(0, 2, 1)       # → (batch, time, channels)

        x, _ = self.lstm(x)

        x = x[:, -1, :]              # last timestep

        x = F.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))

        return x

In [21]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np

In [41]:
class SensorDataset(Dataset):

    def __init__(self, df, window_size=20, shift=10):
        self.data = df.iloc[:,0].values.astype(np.float32)
        self.labels = df.iloc[:,-1].values.astype(np.float32)

        self.window = window_size
        self.shift = shift

    def __len__(self):
        return len(self.data) - self.window - self.shift

    def __getitem__(self, idx):

        X = self.data[idx : idx + self.window]

        y = self.labels[idx + self.window + self.shift]

        return torch.tensor(X), torch.tensor(y)
        
        

In [42]:
# df = pd.read_csv("./sensor_dataset.csv")

# dummy_set = SensorDataset(df)
# dummy_loader = DataLoader(dummy_set, batch_size=32, shuffle=True)

df_train = pd.read_csv("./data.csv")
df_test = pd.read_csv("./test.csv")

train_set = SensorDataset(df_train)
test_set = SensorDataset(df_test)

train_loader = DataLoader(train_set, batch_size=16, shuffle=True)
test_loader = DataLoader(test_set, batch_size=16)

In [43]:
model = LineRatePredictor(1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = model.to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

cuda


In [49]:
def train_model(model, train_loader, val_loader, epochs=50):

    for epoch in range(epochs):

        model.train()
        train_loss = 0

        for X_batch, y_batch in train_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            optimizer.zero_grad()

            preds = model(X_batch.unsqueeze(-1)).squeeze()

            loss = criterion(preds, y_batch)

            loss.backward()

            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        # validation
        model.eval()
        val_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():

            for X_batch, y_batch in val_loader:

                X_batch = X_batch.to(device)
                y_batch = y_batch.to(device)

                preds = model(X_batch.unsqueeze(-1)).squeeze()

                loss = criterion(preds, y_batch)

                val_loss += loss.item()

                predicted = (preds > 0.5).float()

                correct += (predicted == y_batch).sum().item()
                total += y_batch.size(0)

        val_loss /= len(val_loader)
        accuracy = correct / total

        print(
            f"Epoch {epoch+1}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {accuracy:.4f}"
        )

In [50]:
train_model(model=model, train_loader=train_loader, val_loader=test_loader)

Epoch 1/50 | Train Loss: 0.0249 | Val Loss: 0.9593 | Val Acc: 0.7571
Epoch 2/50 | Train Loss: 0.0259 | Val Loss: 1.0279 | Val Acc: 0.7429
Epoch 3/50 | Train Loss: 0.0227 | Val Loss: 1.3074 | Val Acc: 0.7286
Epoch 4/50 | Train Loss: 0.0208 | Val Loss: 1.3561 | Val Acc: 0.7286
Epoch 5/50 | Train Loss: 0.0273 | Val Loss: 1.3406 | Val Acc: 0.7286
Epoch 6/50 | Train Loss: 0.0218 | Val Loss: 1.0186 | Val Acc: 0.7857
Epoch 7/50 | Train Loss: 0.0356 | Val Loss: 1.2857 | Val Acc: 0.7286
Epoch 8/50 | Train Loss: 0.0251 | Val Loss: 1.1466 | Val Acc: 0.7571
Epoch 9/50 | Train Loss: 0.0216 | Val Loss: 1.2766 | Val Acc: 0.7429
Epoch 10/50 | Train Loss: 0.0221 | Val Loss: 1.2281 | Val Acc: 0.7429
Epoch 11/50 | Train Loss: 0.0236 | Val Loss: 1.3263 | Val Acc: 0.7286
Epoch 12/50 | Train Loss: 0.0279 | Val Loss: 1.1170 | Val Acc: 0.7714
Epoch 13/50 | Train Loss: 0.0213 | Val Loss: 1.3055 | Val Acc: 0.7286
Epoch 14/50 | Train Loss: 0.0375 | Val Loss: 1.2574 | Val Acc: 0.7429
Epoch 15/50 | Train Loss: 0.0

In [51]:
torch.save(model.state_dict(), "model2.pth")

In [52]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy('/content/model2.pth', '/content/drive/MyDrive/model2.pth')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


'/content/drive/MyDrive/model2.pth'

In [ ]:
!pwd

/content
